In [28]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, precision_recall_curve

df = pd.read_parquet("../data/02_intermediate/model_input_data.parquet")
df = df.drop(columns=["loan_sequence_number"])  # ID, not a feature
df.shape

(50000, 32)

In [29]:
TARGET = "default"
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)
# stratify=y is important given the ~5.4% default rate — keeps the same imbalance across splits
# val is carved out of train so we can pick a decision threshold without touching test
y_train.value_counts(normalize=True)

default
0    0.946219
1    0.053781
Name: proportion, dtype: float64

In [30]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
print(f"{len(cat_cols)} categorical, {len(num_cols)} numeric")

11 categorical, 20 numeric


In [31]:
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

In [32]:
X_train_hgb = X_train.copy()
X_val_hgb = X_val.copy()
X_test_hgb = X_test.copy()
for col in cat_cols:
    X_train_hgb[col] = X_train_hgb[col].astype("category")
    X_val_hgb[col] = X_val_hgb[col].astype("category")
    X_test_hgb[col] = X_test_hgb[col].astype("category")

In [33]:
# MLflow setup
mlflow.set_experiment("mortgage_default_baseline_models")
mlflow.sklearn.autolog(log_models=True, log_input_examples=False)
# autolog handles most param/metric logging automatically
# domain-specific metrics on top (recall on the default class matters most here,
# per the project doc: false negatives — missed defaults — cost more than false positives)

In [34]:
def evaluate_and_log(model, X_te, y_te, run_name, threshold=0.5, extra_params=None):
    with mlflow.start_run(run_name=run_name):
        if extra_params:
            mlflow.log_params(extra_params)
        mlflow.log_param("decision_threshold", threshold)
        proba = model.predict_proba(X_te)[:, 1]
        preds = (proba >= threshold).astype(int)
        metrics = {
            "test_auc_roc": roc_auc_score(y_te, proba),
            "test_f1_default": f1_score(y_te, preds),
            "test_recall_default": recall_score(y_te, preds),
            "test_precision_default": precision_score(y_te, preds),
        }
        mlflow.log_metrics(metrics)
        return metrics

In [35]:
def get_best_threshold_f1(model, X_val, y_val):
    # picks the threshold that maximizes F1 on the default class, using the
    # validation set (not test) so the choice doesn't leak into the final numbers
    proba_val = model.predict_proba(X_val)[:, 1]
    prec, rec, thresh = precision_recall_curve(y_val, proba_val)
    f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
    return thresh[f1_scores[:-1].argmax()]

In [36]:
def get_best_threshold(model, X_val, y_val, recall_floor=0.70):
    # picks the highest threshold that still keeps recall on the default class
    # at or above recall_floor — maximizes precision subject to that constraint,
    # since precision tends to increase as threshold increases while recall decreases
    proba_val = model.predict_proba(X_val)[:, 1]
    prec, rec, thresh = precision_recall_curve(y_val, proba_val)
    # rec[i] corresponds to thresh[i] for i < len(thresh); rec's last entry (recall=0)
    # has no associated threshold, so it's excluded here
    valid = rec[:-1] >= recall_floor
    if not valid.any():
        # no threshold on the curve reaches the floor — fall back to the lowest
        # threshold, which gives the highest achievable recall for this model
        return thresh[0]
    return thresh[valid].max()

In [37]:
# logistic regression
log_reg = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
log_reg.fit(X_train, y_train)
best_t_lr = get_best_threshold(log_reg, X_val, y_val)
evaluate_and_log(log_reg, X_test, y_test, "logistic_regression", threshold=best_t_lr)

2026/06/27 00:12:58 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '65c7736bce3d4e0baf60655b232db270', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


2026/06/27 00:12:59 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\mariana\Projects\mlops-project\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\mariana\Projects\mlops-project\.venv\Lib\site-packages\sklearn\impute\_base.py:598: UserWarning: Skipping features without any o

{'test_auc_roc': np.float64(0.8249379831986919),
 'test_f1_default': np.float64(0.2585034013605442),
 'test_recall_default': np.float64(0.7063197026022305),
 'test_precision_default': np.float64(0.1582014987510408)}

In [38]:
# balanced random forest
rf = Pipeline([
    ("prep", preprocessor),
    ("clf", BalancedRandomForestClassifier(
        sampling_strategy="all",
        replacement=True,
        bootstrap=False,
        random_state=42,
    )),
])
rf.fit(X_train, y_train)
best_t_rf = get_best_threshold(rf, X_val, y_val)
evaluate_and_log(rf, X_test, y_test, "balanced_random_forest", threshold=best_t_rf)

2026/06/27 00:13:06 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '753148687b214271b862586056885f4a', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/06/27 00:13:07 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\mariana\Projects\mlops-project\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With

{'test_auc_roc': np.float64(0.8118256630513445),
 'test_f1_default': np.float64(0.24298469387755103),
 'test_recall_default': np.float64(0.70817843866171),
 'test_precision_default': np.float64(0.14665127020785218)}

In [39]:
# hist gradient boosting classifier
sw = compute_sample_weight(class_weight="balanced", y=y_train)

hgb = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=42)
hgb.fit(X_train_hgb, y_train, sample_weight=sw)
best_t_hgb = get_best_threshold(hgb, X_val_hgb, y_val)
evaluate_and_log(hgb, X_test_hgb, y_test, "hist_gradient_boosting", threshold=best_t_hgb)

2026/06/27 00:13:21 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '1554023ebd254b3bb821714fa0e1254d', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/06/27 00:13:22 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\mariana\Projects\mlops-project\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With

{'test_auc_roc': np.float64(0.8277904221071333),
 'test_f1_default': np.float64(0.2539464653397392),
 'test_recall_default': np.float64(0.6877323420074349),
 'test_precision_default': np.float64(0.15572390572390574)}

In [40]:
runs = mlflow.search_runs(experiment_names=["mortgage_default_baseline_models"])
runs[["tags.mlflow.runName", "metrics.test_auc_roc", "metrics.test_recall_default",
      "metrics.test_f1_default", "metrics.test_precision_default", "params.decision_threshold"]]

,tags.mlflow.runName,metrics.test_auc_roc,metrics.test_recall_default,metrics.test_f1_default,metrics.test_precision_default,params.decision_threshold
0,hist_gradient_boosting,0.827790,0.687732,0.253946,0.155724,0.5143699496342516
1,merciful-hawk-23,NaN,NaN,NaN,NaN,None
2,balanced_random_forest,0.811826,0.708178,0.242985,0.146651,0.45
3,colorful-goose-884,NaN,NaN,NaN,NaN,None
4,logistic_regression,0.824938,0.706320,0.258503,0.158201,0.5629304423098129
5,sassy-crane-398,NaN,NaN,NaN,NaN,None
6,hist_gradient_boosting,0.827790,0.477695,0.308523,0.227837,0.6886216137977585
7,lyrical-frog-513,NaN,NaN,NaN,NaN,None
8,balanced_random_forest,0.811826,0.414498,0.285349,0.217561,0.6
9,orderly-shark-482,NaN,NaN,NaN,NaN,None
